# Day 1: First Agent — "What's the capital of France?"

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> An agent is a model wrapped in a job description. That's it.

Before we add tools, memory, routing, or multi-agent complexity, we need to prove
the simplest possible case works. Today we build a geography assistant in **all three
frameworks** and see how each one handles the most basic possible agent.

## What You'll Build

- An agent that answers: *"What is the capital of France? What is the capital of Japan?"*
- The same question, three frameworks, side by side
- Real output from each — no simulations, no mocks

## Why This Matters

A question this simple reveals everything about a framework's design philosophy:
- How many objects do you create?
- How much ceremony (boilerplate) is required?
- How readable is the code for a beginner?

**The answer to all three questions is your first real framework comparison.**

In [1]:
# ── Setup ───────────────────────────────────────────────
# This cell checks dependencies, verifies Ollama, and imports all config.
# Run this first — every notebook in this course has this cell.

import sys
sys.path.insert(0, "..")

from shared import check_and_report
all_ok = check_and_report()

if all_ok:
    from shared import (
        print_config,
        disable_openai_tracing,
        get_openai_agents_model,
        get_langgraph_model,
        get_crewai_llm,
    )

    # OpenAI Agents SDK tries to phone home by default.
    # With Ollama, we must disable tracing.
    disable_openai_tracing()

    print("=" * 50)
    print("DAY 1 SETUP COMPLETE")
    print("=" * 50)
    print_config()
    print("=" * 50)
else:
    print("\nFix missing packages before continuing.")

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (8 model(s) available)

DAY 1 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Question

> **"What is the capital of France? What is the capital of Japan?"**

A question so simple it seems silly. But watch how each framework handles it
differently. That difference IS the lesson.

| | OpenAI Agents SDK | LangGraph | CrewAI |
|---|---|---|---|
| **Mental model** | "Give a worker a job" | "Draw a flowchart" | "Hire a team member" |
| **Ceremony level** | Low | Medium-High | Medium |

Let's see this in code.

---
## Framework 1: OpenAI Agents SDK

**Mental model:** An AI worker with a job description.

The OpenAI Agents SDK is the simplest way to create an agent. You need:
1. An `Agent` (name + instructions + model)
2. A `Runner` (executes the agent)

That's it. Two objects. Let's see it.

In [5]:
# ── Framework 1: OpenAI Agents SDK ─────────────────────
# "An AI worker with a job description"
#
# The ceremony: Agent() + Runner.run_sync() — that's it.

from agents import Agent, Runner

model = get_openai_agents_model()

agent = Agent(
    name="Geography Assistant",
    instructions="You are a geography expert. Answer concisely. Name the capitals clearly.",
    model=model,
)

question = "What is the capital of France? What is the capital of Japan?"

print("Running OpenAI Agents SDK...")
result = await Runner.run(starting_agent=agent, input=question)

print()
print("=" * 60)
print("OPENAI AGENTS SDK — OUTPUT")
print("=" * 60)
print(result.final_output)
print("=" * 60)
print(f"Response length: {len(result.final_output)} characters")

Running OpenAI Agents SDK...

OPENAI AGENTS SDK — OUTPUT
The capital of France is Paris.
The capital of Japan is Tokyo.
Response length: 62 characters


---
## Framework 2: LangGraph

**Mental model:** A stateful flowchart.

LangGraph gives you explicit control over every step. Even for a simple question,
the framework thinks in terms of **nodes** (functions that do work) and **edges**
(paths between nodes).

For today, we use the **prebuilt** `create_react_agent` — LangGraph's shortcut for
simple agents. Later in the course, we'll build graphs manually.

In [6]:
# ── Framework 2: LangGraph ──────────────────────────────
# "A stateful flowchart"
#
# The ceremony: create_react_agent() + invoke()
# (Using the prebuilt shortcut — we'll build manual graphs on Day 9)

from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

model = get_langgraph_model()

# create_react_agent builds a ReAct-style agent with optional tools.
# For today, no tools — just the LLM reasoning about the question.
graph_agent = create_react_agent(model, tools=[])

question = "What is the capital of France? What is the capital of Japan?"

print("Running LangGraph...")
result = graph_agent.invoke(
    {"messages": [HumanMessage(content=question)]}
)

# Extract the last message (the agent's answer)
last_message = result["messages"][-1].content

print()
print("=" * 60)
print("LANGGRAPH — OUTPUT")
print("=" * 60)
print(last_message)
print("=" * 60)
print(f"Response length: {len(last_message)} characters")
print(f"Messages in state: {len(result['messages'])}")

/var/folders/s0/2pw05fm942n5t1b5j7zzd_5m0000gn/T/ipykernel_66978/14178523.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph_agent = create_react_agent(model, tools=[])


Running LangGraph...

LANGGRAPH — OUTPUT
The capital of France is Paris.
The capital of Japan is Tokyo.
Response length: 62 characters
Messages in state: 2


---
## Framework 3: CrewAI

**Mental model:** A role-based team member.

CrewAI thinks in terms of **roles, goals, and backstories**. Even a single agent
gets a job title, a goal, and a backstory. It sounds like extra work for a simple
question — and it is. But this role-based thinking becomes powerful when you build
multi-agent teams later in the course.

You need: `Agent` + `Task` + `Crew` + `kickoff()`

In [8]:
# ── Framework 3: CrewAI ────────────────────────────────
# "A role-based team member"
#
# The ceremony: Agent() + Task() + Crew() + kickoff()
# CrewAI requires: role, goal, backstory (even for a single agent)

from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

geographer = Agent(
    role="Geography Expert",
    goal="Answer geography questions accurately and concisely",
    backstory="You are a geography professor with 20 years of experience. "
                 "You love making geography accessible to everyone.",
    llm=llm,
    verbose=True,
)

geo_task = Task(
    description="What is the capital of France? What is the capital of Japan?",
    expected_output="A concise answer naming both capitals clearly.",
    agent=geographer,
)

crew = Crew(
    agents=[geographer],
    tasks=[geo_task],
    process=Process.sequential,
    verbose=True,
)

print("Running CrewAI...")
result = await crew.kickoff_async()

print()
print("=" * 60)
print("CREWAI — OUTPUT")
print("=" * 60)
print(result)
print("=" * 60)
print(f"Response length: {len(str(result))} characters")

Running CrewAI...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ce26c24a-4f30-4ba2-9bac-10a4252e6a01                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is the capital of France? What is the capital of Japan?                                             │
│  ID: 9728e9a8-3833-4030-97e2-5340976b474b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Geography Expert                                                                                        │
│                                                                                                                 │
│  Task: What is the capital of France? What is the capital of Japan?                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Geography Expert                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The capital of France is Paris. The capital of Japan is Tokyo.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: What is the capital of France? What is the capital of Japan?                                             │
│  Agent: Geography Expert                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


CREWAI — OUTPUT
The capital of France is Paris. The capital of Japan is Tokyo.
Response length: 62 characters


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ce26c24a-4f30-4ba2-9bac-10a4252e6a01                                                                       │
│  Final Output: The capital of France is Paris. The capital of Japan is Tokyo.                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Side-by-Side Comparison

You just ran the same question through three frameworks. Here's what you observed:

In [9]:
# ── Framework Comparison ──────────────────────────────
# This comparison is based on the code above, not opinions.

comparison = """
┌─────────────────────┬──────────────────┬──────────────────┬──────────────────┐
| Aspect              | OpenAI Agents SDK| LangGraph         | CrewAI            |
├─────────────────────┼──────────────────┼──────────────────┼──────────────────┤
| Objects to create   | Agent + Runner   | Graph (prebuilt) | Agent+Task+Crew   |
| Lines of code       | ~5               | ~5 (prebuilt)    | ~12              |
| Required knowledge  | Minimal          | Graph concepts   | Role/goal model   |
| How you describe it | "Give a worker   | "Draw a flowchart"| "Hire a team      |
|                     |  a job"         |                  |   member"        |
| Verbose by default  | No               | No               | Yes              |
| Best for            | Quick prototypes | Complex workflows| Role-based teams  |
└─────────────────────┴──────────────────┴──────────────────┴──────────────────┘
"""
print(comparison)

print("KEY INSIGHT:")
print("- OpenAI SDK: least ceremony, fastest to start")
print("- LangGraph: more ceremony, but gives you explicit control over every step")
print("- CrewAI: role-based thinking feels natural for multi-agent teams")
print()
print("The extra ceremony in LangGraph and CrewAI isn't waste — it's control surface.")
print("When you need branching, loops, or teams, those extra lines earn their keep.")


┌─────────────────────┬──────────────────┬──────────────────┬──────────────────┐
| Aspect              | OpenAI Agents SDK| LangGraph         | CrewAI            |
├─────────────────────┼──────────────────┼──────────────────┼──────────────────┤
| Objects to create   | Agent + Runner   | Graph (prebuilt) | Agent+Task+Crew   |
| Lines of code       | ~5               | ~5 (prebuilt)    | ~12              |
| Required knowledge  | Minimal          | Graph concepts   | Role/goal model   |
| How you describe it | "Give a worker   | "Draw a flowchart"| "Hire a team      |
|                     |  a job"         |                  |   member"        |
| Verbose by default  | No               | No               | Yes              |
| Best for            | Quick prototypes | Complex workflows| Role-based teams  |
└─────────────────────┴──────────────────┴──────────────────┴──────────────────┘

KEY INSIGHT:
- OpenAI SDK: least ceremony, fastest to start
- LangGraph: more ceremony, but gives you

---
## Key Takeaway

An agent is a model wrapped in a job description. The three frameworks differ only
in **how much ceremony** they require to express that description:

- **OpenAI Agents SDK** — "Here's the job." (5 lines)
- **LangGraph** — "Here's the flowchart for doing the job." (5-25 lines)
- **CrewAI** — "Here's who's doing the job and why." (12 lines)

The real skill isn't picking a framework. It's knowing what each one trades away
to give you its strength.

---

## Exercise

1. Change the question to something from YOUR domain (e.g., "What is the difference
   between a list and a tuple in Python?")
2. Run all three frameworks again.
3. Which one felt most natural for your use case?
4. **Write down your answer** — you'll need it on later when we decide when to
   use which framework.